# SingleStore Database Rebuild Kit: owner steps

Author: John Bergamini

One cell per admin step. Run them in order inside the outage window, after the kit printed the step as pending. Each cell prints a JSON result block; save it to a file and record the gate with `s2rebuild gate <ID> --pass --note TEXT --evidence <file>`. The O2 result carries the pipeline capture taken after the stop, which the kit imports for O9; after the O3 cell run `s2rebuild step O3 --redo`, which runs the write probe and records freeze_time.

Run {{RUN_ID}} for database {{SOURCE}}. Generated {{GENERATED_UTC}}.

In [ ]:
{{OWNERLIB}}

import json
RUN = json.loads({{RUN_JSON}})
BLOCKS = json.loads({{BLOCKS_JSON}})
print('run', RUN['run_id'], 'database', RUN['source'], 'blocks', sorted(BLOCKS))


In [ ]:
import singlestoredb as s2
from singlestoredb.notebook import portal
if RUN.get('workspace_id'):
    assert str(portal.workspace_id) == str(RUN['workspace_id']), 'this notebook is attached to workspace %s, the run expects %s' % (portal.workspace_id, RUN['workspace_id'])
url = portal.connection_url
host = url.split('@')[-1].split(':')[0].split('/')[0]
assert host == RUN['host'], 'this notebook is attached to %s, the run expects %s; select the right workspace' % (host, RUN['host'])


def connect():
    """A new connection to the attached workspace (the DETACH escalation needs a second one)."""
    return s2.connect(url, results_type='dicts', autocommit=True)


conn = connect()
cur = conn.cursor()
cur.execute('SELECT CURRENT_USER() AS u, @@memsql_version AS v')
print(_rows(cur))


In [ ]:
# O2 stop pipelines; the result carries the capture after the stop: save it and run s2rebuild gate O2 --pass --note TEXT --evidence <file>
result = run_block(cur, BLOCKS['O2'], RUN)


In [ ]:
# O3 freeze (DETACH then ATTACH READ ONLY); afterwards run s2rebuild step O3 --redo for the write probe and freeze_time
result = run_block(cur, BLOCKS['O3'], RUN, connect=connect)
if result['status'] == 'FAILED at check source attached READWRITE' and result['checks'][-1]['observed'] == 'no rows':
    # No database carries the source name: an earlier run stopped between DETACH and ATTACH. O3-reentry checks that the
    # recorded storage is detached and the source and replacement names are free, then starts at the ATTACH; no DETACH again.
    result = run_block(cur, BLOCKS['O3-reentry'], RUN)


In [ ]:
# O9 pipelines: each CREDENTIALS JSON is read with getpass (never saved in the notebook), keyed by pipeline name or source type
import getpass
credentials = {key: getpass.getpass('CREDENTIALS JSON for %s: ' % key) for key in RUN['credential_keys']}
result = run_block(cur, BLOCKS['O9'], RUN, credentials=credentials)
del credentials


In [ ]:
# O10 switch forward (S1 to S4, two entries at the end); the result carries the retained and live records for s2rebuild gate O10 --evidence
result = run_block(cur, BLOCKS['O10'], RUN, connect=connect)


In [ ]:
# O10 rollback, only before O11 released writers: O10-rollback (R1, R2a) before S2b ran, O10-rollback-retained (R1, R2b) after
# result = run_block(cur, BLOCKS['O10-rollback'], RUN)


In [ ]:
# O11 release: start pipelines, parallelism
result = run_block(cur, BLOCKS['O11'], RUN)


In [ ]:
# C3 end state (two entries)
# result = run_block(cur, BLOCKS['C3'], RUN)


In [ ]:
# C4 remove the migration account
# result = run_block(cur, BLOCKS['C4'], RUN)
